In [1]:
import pandas as pd
import numpy as np
import os, json, joblib, warnings, gc
from tqdm import tqdm
warnings.filterwarnings('ignore')

# Try GPU-accelerated cuML (available on Kaggle T4 notebooks)
try:
    from cuml.ensemble import RandomForestClassifier as cuRF
    from cuml.linear_model import LogisticRegression as cuLR
    from cuml import ForestInference
    import cudf
    USE_GPU = True
    print('✓ Using cuML (GPU-accelerated)')
except ImportError:
    USE_GPU = False
    print('✗ cuML not available, using sklearn (CPU)')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

✓ Using cuML (GPU-accelerated)


In [3]:
# ── Kaggle Paths ──
DATASET_DIRECTORY = '/kaggle/input/datasets/madhavmalhotra/unb-cic-iot-dataset/wataiData/csv/CICIoT2023/'
OUTPUT_DIR = '/kaggle/working/'

df_sets = sorted([k for k in os.listdir(DATASET_DIRECTORY) if k.endswith('.csv')])
training_sets = df_sets[:int(len(df_sets)*0.8)]
test_sets = df_sets[int(len(df_sets)*0.8):]
print(f'Total: {len(df_sets)}, Train: {len(training_sets)}, Test: {len(test_sets)}')

Total: 169, Train: 135, Test: 34


In [4]:
X_columns = [
    'flow_duration', 'Header_Length', 'Protocol Type', 'Duration',
    'Rate', 'Srate', 'Drate', 'fin_flag_number', 'syn_flag_number',
    'rst_flag_number', 'psh_flag_number', 'ack_flag_number',
    'ece_flag_number', 'cwr_flag_number', 'ack_count',
    'syn_count', 'fin_count', 'urg_count', 'rst_count',
    'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP',
    'UDP', 'DHCP', 'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min',
    'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Magnitue',
    'Radius', 'Covariance', 'Variance', 'Weight',
]
y_column = 'label'

LIVE_FEATURE_COLS = ['Rate', 'Srate', 'Protocol Type', 'Variance']
LIVE_FEATURE_NAMES = ['pkt_rate', 'byte_rate', 'unique_ports', 'port_entropy']

# Preview
sample = pd.read_csv(DATASET_DIRECTORY + training_sets[0], nrows=3)
print('Labels:', sample[y_column].values)
sample.head()

Labels: ['DDoS-RSTFINFlood' 'DoS-TCP_Flood' 'DDoS-ICMP_Flood']


,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
0,0.0,54.00,6.00,64.0,0.329807,0.329807,0.0,1.0,0.0,1.0,...,0.000000,54.00,8.334383e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,DDoS-RSTFINFlood
1,0.0,57.04,6.33,64.0,4.290556,4.290556,0.0,0.0,0.0,0.0,...,2.822973,57.04,8.292607e+07,9.5,10.464666,4.010353,160.987842,0.05,141.55,DoS-TCP_Flood
2,0.0,0.00,1.00,64.0,33.396799,33.396799,0.0,0.0,0.0,0.0,...,0.000000,42.00,8.312799e+07,9.5,9.165151,0.000000,0.000000,0.00,141.55,DDoS-ICMP_Flood


In [5]:
print('Collecting benign samples for Isolation Forest...')
benign_chunks = []
total_rows = 0

for f in tqdm(training_sets):
    d = pd.read_csv(DATASET_DIRECTORY + f, usecols=LIVE_FEATURE_COLS + [y_column])
    total_rows += len(d)
    b = d[d[y_column] == 'BenignTraffic'][LIVE_FEATURE_COLS]
    if len(b) > 0:
        benign_chunks.append(b)
    del d; gc.collect()

X_benign = pd.concat(benign_chunks, ignore_index=True)
X_benign.replace([np.inf, -np.inf], np.nan, inplace=True)
X_benign.fillna(0, inplace=True)
del benign_chunks; gc.collect()

print(f'Total rows: {total_rows:,}, Benign: {len(X_benign):,}')

100%|██████████| 135/135 [03:30<00:00,  1.56s/it]

Total rows: 36,346,418, Benign: 854,873


In [6]:
live_scaler = StandardScaler()
X_benign_scaled = live_scaler.fit_transform(X_benign)

print('Training Isolation Forest...')
if_model = IsolationForest(
    n_estimators=200,
    max_samples=0.8 if len(X_benign_scaled) > 1000 else 'auto',
    contamination=0.05,
    random_state=42,
    n_jobs=-1,
)
if_model.fit(X_benign_scaled)
print('Done!')

Training Isolation Forest...
Done!


In [7]:
print('Evaluating IF on test set...')
y_true_all, y_pred_all = [], []

for f in tqdm(test_sets):
    d = pd.read_csv(DATASET_DIRECTORY + f, usecols=LIVE_FEATURE_COLS + [y_column])
    X_t = d[LIVE_FEATURE_COLS].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_t_scaled = live_scaler.transform(X_t)
    y_true = (d[y_column] != 'BenignTraffic').astype(int)
    y_pred = np.where(if_model.predict(X_t_scaled) == 1, 0, 1)
    y_true_all.extend(y_true.values)
    y_pred_all.extend(y_pred)
    del d; gc.collect()

if_acc = accuracy_score(y_true_all, y_pred_all)
if_prec = precision_score(y_true_all, y_pred_all, zero_division=0)
if_rec = recall_score(y_true_all, y_pred_all, zero_division=0)
if_f1 = f1_score(y_true_all, y_pred_all, zero_division=0)

print(f'\n=== Isolation Forest ===')
print(f'Accuracy:  {if_acc:.4f}')
print(f'Precision: {if_prec:.4f}')
print(f'Recall:    {if_rec:.4f}')
print(f'F1 Score:  {if_f1:.4f}')

Evaluating IF on test set...


100%|██████████| 34/34 [03:51<00:00,  6.82s/it]



=== Isolation Forest ===
Accuracy:  0.9128
Precision: 0.9987
Recall:    0.9120
F1 Score:  0.9533


In [8]:
# Save model artifacts
joblib.dump(if_model, os.path.join(OUTPUT_DIR, 'live_if_model.pkl'))
joblib.dump(live_scaler, os.path.join(OUTPUT_DIR, 'live_scaler.pkl'))

meta = {
    'model_type': 'IsolationForest',
    'features': LIVE_FEATURE_COLS,
    'live_feature_names': LIVE_FEATURE_NAMES,
    'best_params': {'n_estimators': 200, 'contamination': 0.05, 'max_samples': 0.8},
    'accuracy': round(if_acc, 4),
    'precision': round(if_prec, 4),
    'recall': round(if_rec, 4),
    'f1_score': round(if_f1, 4),
    'training_samples': int(len(X_benign_scaled)),
    'dataset': 'CICIoT2023 (WATAI)',
    'total_files': len(df_sets),
}
with open(os.path.join(OUTPUT_DIR, 'live_model_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved: live_if_model.pkl, live_scaler.pkl, live_model_meta.json')
del X_benign, X_benign_scaled; gc.collect()

Saved: live_if_model.pkl, live_scaler.pkl, live_model_meta.json


33

In [9]:
full_scaler = StandardScaler()
print('Fitting scaler...')
for f in tqdm(training_sets):
    d = pd.read_csv(DATASET_DIRECTORY + f)
    d[X_columns] = d[X_columns].replace([np.inf, -np.inf], np.nan).fillna(0)
    full_scaler.partial_fit(d[X_columns])
    del d; gc.collect()
print('Scaler fitted.')

Fitting scaler...


100%|██████████| 135/135 [04:04<00:00,  1.81s/it]

Scaler fitted.


In [10]:
lr_34 = LogisticRegression(n_jobs=-1, max_iter=1000, warm_start=True)

print('Training LR (34 classes)...')
for f in tqdm(training_sets):
    d = pd.read_csv(DATASET_DIRECTORY + f)
    d[X_columns] = d[X_columns].replace([np.inf, -np.inf], np.nan).fillna(0)
    d[X_columns] = full_scaler.transform(d[X_columns])
    lr_34.fit(d[X_columns], d[y_column])
    del d; gc.collect()
print('Done!')

Training LR (34 classes)...


  5%|▌         | 7/135 [14:32<4:25:49, 124.61s/it]


KeyboardInterrupt: 

In [ ]:
y_test, preds = [], []
for f in tqdm(test_sets):
    d = pd.read_csv(DATASET_DIRECTORY + f)
    d[X_columns] = d[X_columns].replace([np.inf, -np.inf], np.nan).fillna(0)
    d[X_columns] = full_scaler.transform(d[X_columns])
    y_test += list(d[y_column].values)
    preds += list(lr_34.predict(d[X_columns]))
    del d; gc.collect()

print(f'\n===== LR (34 classes) =====')
print(f'accuracy:  {accuracy_score(preds, y_test):.4f}')
print(f'recall:    {recall_score(preds, y_test, average="macro"):.4f}')
print(f'precision: {precision_score(preds, y_test, average="macro"):.4f}')
print(f'f1_score:  {f1_score(preds, y_test, average="macro"):.4f}')

In [ ]:
dict_7classes = {
    'DDoS-RSTFINFlood': 'DDoS', 'DDoS-PSHACK_Flood': 'DDoS',
    'DDoS-SYN_Flood': 'DDoS', 'DDoS-UDP_Flood': 'DDoS',
    'DDoS-TCP_Flood': 'DDoS', 'DDoS-ICMP_Flood': 'DDoS',
    'DDoS-SynonymousIP_Flood': 'DDoS', 'DDoS-ACK_Fragmentation': 'DDoS',
    'DDoS-UDP_Fragmentation': 'DDoS', 'DDoS-ICMP_Fragmentation': 'DDoS',
    'DDoS-SlowLoris': 'DDoS', 'DDoS-HTTP_Flood': 'DDoS',
    'DoS-UDP_Flood': 'DoS', 'DoS-SYN_Flood': 'DoS',
    'DoS-TCP_Flood': 'DoS', 'DoS-HTTP_Flood': 'DoS',
    'Mirai-greeth_flood': 'Mirai', 'Mirai-greip_flood': 'Mirai',
    'Mirai-udpplain': 'Mirai',
    'Recon-PingSweep': 'Recon', 'Recon-OSScan': 'Recon',
    'Recon-PortScan': 'Recon', 'VulnerabilityScan': 'Recon',
    'Recon-HostDiscovery': 'Recon',
    'DNS_Spoofing': 'Spoofing', 'MITM-ArpSpoofing': 'Spoofing',
    'BenignTraffic': 'Benign',
    'BrowserHijacking': 'Web', 'Backdoor_Malware': 'Web',
    'XSS': 'Web', 'Uploading_Attack': 'Web',
    'SqlInjection': 'Web', 'CommandInjection': 'Web',
    'DictionaryBruteForce': 'BruteForce',
}

In [ ]:
lr_8 = LogisticRegression(n_jobs=-1, max_iter=1000, warm_start=True)

print('Training LR (8 classes)...')
for f in tqdm(training_sets):
    d = pd.read_csv(DATASET_DIRECTORY + f)
    d[X_columns] = d[X_columns].replace([np.inf, -np.inf], np.nan).fillna(0)
    d[X_columns] = full_scaler.transform(d[X_columns])
    d[y_column] = [dict_7classes[k] for k in d[y_column]]
    lr_8.fit(d[X_columns], d[y_column])
    del d; gc.collect()
print('Done!')

In [ ]:
y_test, preds = [], []
for f in tqdm(test_sets):
    d = pd.read_csv(DATASET_DIRECTORY + f)
    d[X_columns] = d[X_columns].replace([np.inf, -np.inf], np.nan).fillna(0)
    d[X_columns] = full_scaler.transform(d[X_columns])
    d[y_column] = [dict_7classes[k] for k in d[y_column]]
    y_test += list(d[y_column].values)
    preds += list(lr_8.predict(d[X_columns]))
    del d; gc.collect()

print(f'\n===== LR (8 classes) =====')
print(f'accuracy:  {accuracy_score(preds, y_test):.4f}')
print(f'recall:    {recall_score(preds, y_test, average="macro"):.4f}')
print(f'precision: {precision_score(preds, y_test, average="macro"):.4f}')
print(f'f1_score:  {f1_score(preds, y_test, average="macro"):.4f}')

In [ ]:
dict_2classes = {k: 'Benign' if k == 'BenignTraffic' else 'Attack' for k in dict_7classes}

lr_2 = LogisticRegression(n_jobs=-1, max_iter=1000, warm_start=True)

print('Training LR (2 classes)...')
for f in tqdm(training_sets):
    d = pd.read_csv(DATASET_DIRECTORY + f)
    d[X_columns] = d[X_columns].replace([np.inf, -np.inf], np.nan).fillna(0)
    d[X_columns] = full_scaler.transform(d[X_columns])
    d[y_column] = [dict_2classes[k] for k in d[y_column]]
    lr_2.fit(d[X_columns], d[y_column])
    del d; gc.collect()
print('Done!')

In [ ]:
y_test, preds = [], []
for f in tqdm(test_sets):
    d = pd.read_csv(DATASET_DIRECTORY + f)
    d[X_columns] = d[X_columns].replace([np.inf, -np.inf], np.nan).fillna(0)
    d[X_columns] = full_scaler.transform(d[X_columns])
    d[y_column] = [dict_2classes[k] for k in d[y_column]]
    y_test += list(d[y_column].values)
    preds += list(lr_2.predict(d[X_columns]))
    del d; gc.collect()

print(f'\n===== LR (2 classes) =====')
print(f'accuracy:  {accuracy_score(preds, y_test):.4f}')
print(f'recall:    {recall_score(preds, y_test, average="macro"):.4f}')
print(f'precision: {precision_score(preds, y_test, average="macro"):.4f}')
print(f'f1_score:  {f1_score(preds, y_test, average="macro"):.4f}')

In [11]:
joblib.dump(full_scaler, os.path.join(OUTPUT_DIR, 'full_scaler.pkl'))

print('\n=== All training complete! ===')
print('Output files in /kaggle/working/:')
print('  - live_if_model.pkl')
print('  - live_scaler.pkl')
print('  - live_model_meta.json')
print('  - full_scaler.pkl')
print('\nDownload these and place in your project\'s models/ folder.')


=== All training complete! ===
Output files in /kaggle/working/:
  - live_if_model.pkl
  - live_scaler.pkl
  - live_model_meta.json
  - full_scaler.pkl

Download these and place in your project's models/ folder.
